# Demo 6: Bouw je eigen RL-toepassing

Jij beschrijft het probleem, de robot vindt de policy met Q-learning. Je past alleen het probleem aan, het leren zelf gebeurt achter de schermen.

| Deel | Wat je doet |
|---|---|
| **A. Definiëren** | toestanden, acties, overgangskansen, beloningen |
| **B. Instellen** | leersnelheid, verkenning, risico, discount |
| **C. Trainen** | een beetje trainen en opslaan, dan laden, verder trainen en weer opslaan |
| **D. Veranderen** | verander de beloningen (of iets anders) en laat de opgeslagen robot opnieuw leren |
| **E. Je eigen probleem** | vervang het voorbeeld door je eigen probleem |

**Het voorbeeld** 3 toestanden (A, B, C) en 3 acties (1, 2, 3). Alles staat in een matrix van wat er na elke actie gebeurt, plus een beloning per toestand en een beloning per actie.

**Zo gebruik je het:** klik één keer op *Run → Run All Cells* om het hele verhaal te zien. Pas daarna een cel aan, voer hem uit en voer de trainingscellen uit die je nodig hebt. Trainen gebruikt altijd de definities en instellingen die je het laatst hebt uitgevoerd.

## Zet de robot aan

Voer deze cel eerst uit. Het leren, controleren en tekenen gebeurt achter de schermen, hier hoef je niets aan te passen.

In [1]:
from rl_demos.build_your_own import *

connect(globals())    # lets the robot read your cells

## A. Definieer het probleem

### 1. Toestanden en acties

Een toestand moet alles bevatten wat ertoe doet voor wat er daarna gebeurt. Hier is elke actie in elke toestand mogelijk.

Als het helpt, lees het voorbeeld als een klant: A is *tevreden*, B is *neutraal* en C is *ontevreden*. Actie 1 is *niets doen*, 2 is *kleine moeite* en 3 is *grote moeite*.

In [2]:
STATES = ['A', 'B', 'C']
ACTIONS = [1, 2, 3]

START = {'A': 1.0}     # where an episode starts, with which probability

### 2. Overgangskansen: een matrix van 3 × 3 × 3

We geven hier aan hoe de omgeving veranderd waar de robot in leeft
De robot ziet deze getallen nooit. Ze zijn de gesimuleerde wereld waarin deze oefent.

In [3]:
TRANSITIONS = [
    #   to:   A    B    C
    [                         # from A
        [0.7, 0.2, 0.1],      #   action 1
        [0.9, 0.1, 0.0],      #   action 2
        [1.0, 0.0, 0.0],      #   action 3
    ],
    [                         # from B
        [0.1, 0.5, 0.4],
        [0.5, 0.4, 0.1],
        [0.8, 0.2, 0.0],
    ],
    [                         # from C
        [0.0, 0.1, 0.9],
        [0.1, 0.4, 0.5],
        [0.5, 0.4, 0.1],
    ],
]

show_diagram()      # draws these probabilities: one picture per action

### 3. Beloningen: een beloning per toestand en een beloning per actie

De robot wil zo veel mogelijk beloning verzamelen. Elke stap levert twee dingen op:
- een vaste beloning voor elke stap die je in een toestand bent
- de beloning van de actie die je kiest (moeite kost iets, dus die is negatief)


In [ ]:
STATE_REWARDS = [10, 8, 0]      # every step in A, B, C
ACTION_REWARDS = [0, -4, -9]    # action 1, 2, 3

REWARDS = combine_rewards(ACTION_REWARDS, STATE_REWARDS)

show_diagram()      # the same picture, now with the rewards on the arrows

## B. Hoe de robot leert

| Instelling | Laag | Hoog |
|---|---|---|
| **Leersnelheid** | langzaam maar stabiel | snel maar onrustig |
| **Verkenning** | vindt misschien nooit iets beters, merkt veranderingen traag op | blijft dingen proberen waarvan bekend is dat ze slecht zijn |
| **Risico** | −1: verliezen doen dubbel zoveel pijn (voorzichtig) | 1: verliezen worden genegeerd (roekeloos) |
| **Discount** | alleen de volgende beloning telt | de lange termijn telt |

Risico verandert alleen wat de robot *voelt* tijdens het leren. Hij wordt altijd beoordeeld op de echte beloningen.

In [5]:
LEARNING_RATE = 0.02   # above 0, up to 1
EXPLORATION = 0.2      # 0 to 1: how often it tries a random action
RISK = 0.0             # -1 (cautious) to 1 (reckless), 0 = neutral
DISCOUNT = 0.9         # 0 up to, not including, 1
MAX_STEPS = 50         # steps per practice episode
SEED = 0               # change it to see whether your result was luck

### Controleer je probleem

Voer dit uit na elke wijziging in deel A. Het noemt fouten, zoals een regel die niet optelt tot 1 of een matrix van de verkeerde grootte, en toont beide matrices als één tabel.

In [6]:
show_problem()

state,action,reward,to A,to B,to C
A,1,0,70%,20%,10%
A,2,-4,90%,10%,0%
A,3,-9,100%,0%,0%
B,1,-2,10%,50%,40%
B,2,-6,50%,40%,10%
B,3,-11,80%,20%,0%
C,1,-10,0%,10%,90%
C,2,-14,10%,40%,50%
C,3,-19,50%,40%,10%


## C. Trainen, opslaan, verder trainen

### 4. Begin met een lege robot

Voer deze cel opnieuw uit als je wilt dat hij alles vergeet.

In [7]:
robot = new_robot()

### 5. Train tot een bepaald punt en sla op

Maar een beetje oefening, dus verwacht fouten. Het bestand bevat de policy en de Q-matrix: de Q-matrix heb je nodig om verder te trainen.

In [8]:
train(robot, episodes=20, save_as='v1')
show_policy(robot)

Trained 20 episodes (0 -> 20 in total)
Saved saved_policies/v1.json (replaced the older one)


state,1,2,3,robot does,perfect policy
A,-5.9417 tries,-5.774 tries,-6.036 tries,2,1
B,-8.3124 tries,-8.276 tries,-8.344 tries,2,2
C,-16.292 tries,-16.675 tries,-16.462 tries,1,3


### 6. Laad hem en train verder

Wil je nog verder, verander dan de namen: laad `'v2'` en sla op als `'v2b'`.

In [9]:
robot = load('v1')
train(robot, episodes=5000, save_as='v2')
show_policy(robot)
show_learning({'robot': robot})

Loaded v1 (20 episodes of experience)
Trained 5,000 episodes (20 -> 5,020 in total)
Saved saved_policies/v2.json (replaced the older one)


state,1,2,3,robot does,perfect policy
A,"-31.9138,560 tries","-34.114,080 tries","-38.210,904 tries",1,1
B,"-41.55,275 tries","-40.350,598 tries","-41.85,573 tries",2,2
C,"-56.42,169 tries","-54.72,523 tries","-52.621,318 tries",3,3


robot,at start,highest,at end
robot,-54,-31,-31


In [10]:
# Watch the policy at work (run again for another episode)
watch(robot)

step,state,action,reward,next state,total
1,A,1,0,A,0
2,A,1,0,A,0
3,A,1,0,A,0
4,A,1,0,B,0
5,B,2,-6,B,-6
6,B,2,-6,C,-12
7,C,3,-19,A,-31
8,A,1,0,A,-31
9,A,1,0,B,-31
10,B,2,-6,A,-37


In [11]:
# The policy as a picture: only the arrows of the action the robot chooses
show_diagram(robot)

## D. De wereld verandert: opnieuw leren

### 7. Verander iets

In het voorbeeld levert een stap in C ineens 8 op in plaats van 0, net zoveel als B. Is moeite dan nog de moeite waard? Je kunt ook elke cel in deel A of B aanpassen en uitvoeren, bijvoorbeeld de kansen of `RISK`.

Deze cel verandert `REWARDS`. Om dat terug te draaien voer je de beloningscel (3) opnieuw uit.

In [ ]:
STATE_REWARDS = [10, 8, 8]      # C: 0 -> 8
REWARDS = combine_rewards(ACTION_REWARDS, STATE_REWARDS)
print('Rewards per state (rows A, B, C) and action (columns 1, 2, 3):', REWARDS)

### 8. Hoe goed is de opgeslagen policy nu?

Hij heeft nog niets over de verandering geleerd.

In [13]:
compare({'v2 (saved before the change)': load('v2')})

Loaded v2 (5,020 episodes of experience)


policy,A,B,C,score
v2 (saved before the change),1,2,3,-24
perfect,1,1,1,-14
random,any,any,any,-48


### 9. Leer opnieuw vanaf de opgeslagen policy

`v2` blijft zoals hij was, de opnieuw getrainde robot wordt opgeslagen als `v3`. In de grafiek zie je de daling op het moment van de verandering, en het herstel.

In [14]:
robot = load('v2')
train(robot, episodes=5000, save_as='v3')
show_policy(robot)
show_learning({'robot': robot})

Loaded v2 (5,020 episodes of experience)
Changed since the last training: REWARDS -> relearning from what it already knew
Trained 5,000 episodes (5,020 -> 10,020 in total)
Saved saved_policies/v3.json (replaced the older one)


state,1,2,3,robot does,perfect policy
A,"-13.8204,494 tries","-17.019,191 tries","-21.415,971 tries",1,1
B,"-18.250,258 tries","-21.057,389 tries","-24.89,158 tries",1,1
C,"-19.2103,286 tries","-22.410,440 tries","-25.530,813 tries",1,1


robot,at start,highest,at end
robot,-54,-14,-14


### 10. Was opnieuw leren beter dan opnieuw beginnen?

Oude kennis geeft vaak een voorsprong, maar na een grote verandering kan hij ook in de weg zitten.

In [15]:
fresh = train(new_robot(), episodes=5000)
compare({'v2 old policy': load('v2'), 'v3 relearned': robot, 'started again': fresh})
show_learning({'relearned': robot, 'started again': fresh}, last_run=True)

Trained 5,000 episodes (0 -> 5,000 in total)
Loaded v2 (5,020 episodes of experience)


policy,A,B,C,score
v2 old policy,1,2,3,-24
v3 relearned,1,1,1,-14
started again,1,1,1,-14
perfect,1,1,1,-14
random,any,any,any,-48


robot,at start,highest,at end
relearned,-24,-14,-14
started again,-48,-14,-14


## E. Bouw nu je eigen probleem

Beantwoord eerst deze vijf vragen op papier:

1. Welke beslissing wordt **steeds opnieuw** genomen? Als hij maar één keer genomen wordt, heb je geen RL nodig.
2. Wat **weet** de robot op dat moment? Dat is de toestand.
3. Wat kan hij **doen**? Dat zijn de acties.
4. Hoe verandert wat hij doet wat er **daarna** gebeurt? Dat zijn de overgangen. Als het niets verandert, heb je ook geen RL nodig.
5. Wat wil je echt, en hoe zou een robot **valsspelen** met jouw beloning?

Vervang daarna cellen 1 tot en met 3 door jouw versie, en verander of leeg het voorbeeld in cel 7. Geef je opgeslagen versies nieuwe namen. Een klein sjabloon met 2 toestanden en 2 acties:

```python
STATES = ['quiet', 'busy']
ACTIONS = ['wait', 'advertise']
START = {'quiet': 1.0}

TRANSITIONS = [
    #  to: quiet busy
    [[0.9, 0.1],      # quiet, wait
     [0.3, 0.7]],     # quiet, advertise
    [[0.4, 0.6],      # busy, wait
     [0.4, 0.6]],     # busy, advertise
]

REWARDS = [
    #  to: quiet busy
    [[1, 5],          # quiet, wait
     [-2, 3]],        # quiet, advertise
    [[1, 5],          # busy, wait
     [-2, 3]],        # busy, advertise
]
```

Als de beloning niet afhangt van waar je terechtkomt, is één getal per actie genoeg: `REWARDS = [[1, -2], [5, 3]]`.

## Voordat je dit echt gebruikt

- **De simulator is een aanname.** De policy is maar zo goed als de kansen die je hebt ingetypt. In de praktijk schat je ze uit data en test je hoe fout ze mogen zijn.
- **Verkennen kost echt geld en kan echte mensen schaden.** Oefen in simulatie, beperk de toegestane acties en houd een mens aan het stuur.
- **Vergelijk met een simpele regel** voordat je RL vertrouwt of ervoor betaalt.
- **Blijf monitoren.** Als de wereld verandert, wordt een opgeslagen policy slechter zonder enige waarschuwing.
- **Tabellen werken niet meer** als er te veel toestanden zijn. Dat is het onderwerp van demo 7.
- **Download je bestanden** uit `saved_policies` op Binder: ze verdwijnen als de sessie stopt.